# muon-encoder: 17M grid on 2x T4

Settings: **Internet on**, **Accelerator: GPU T4 x2**, **Persistence: off**.

Inputs (Add Input → Your Work): the data notebook's output, and — from the second version onwards — this notebook's own previous output, so unfinished runs resume from their checkpoints.

Run with **Save Version → Save & Run All**. The workers stop themselves after `HOURS` so the version finishes inside Kaggle's 12h limit and its output is saved.

In [ ]:
REPO = "https://github.com/Jakevanveen123/muon-encoder"
HOURS = 11.3

import glob, os
DATA = os.path.dirname(glob.glob("/kaggle/input/**/train.bin", recursive=True)[0])
previous = glob.glob("/kaggle/input/**/runs", recursive=True)
PREVIOUS_RUNS = previous[0] if previous else None
print("data:", DATA, os.listdir(DATA))
print("previous runs:", PREVIOUS_RUNS)

In [ ]:
import shutil, subprocess, time, signal

!git clone -q $REPO code
%cd code
!pip install -q -U transformers wandb 2>&1 | tail -1
if PREVIOUS_RUNS:
    shutil.copytree(PREVIOUS_RUNS, "/kaggle/working/runs")
    print("resuming from", os.listdir("/kaggle/working/runs"))

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    os.environ["WANDB_MODE"] = "offline"
    print("no WANDB_API_KEY secret, logging offline")

In [ ]:
COMMON = f"--size 17m --dtype fp16 --data_dir {DATA} --out_dir /kaggle/working/runs"
WORKERS = {
    "cuda:0": [
        "--optimizer muon --phase sweep",
        "--optimizer muon --phase seeds",
        "--optimizer adamw --phase budget --seeds 0",
    ],
    "cuda:1": [
        "--optimizer adamw --phase sweep",
        "--optimizer adamw --phase seeds",
        "--optimizer adamw --phase budget --seeds 1 2",
    ],
}

procs = {}
for device, phases in WORKERS.items():
    script = " && ".join(f"python sweep.py {p} {COMMON} --device {device}" for p in phases)
    log = open(f"/kaggle/working/log_{device.replace(':', '')}.txt", "a")
    procs[device] = subprocess.Popen(script, shell=True, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

In [ ]:
deadline = time.time() + HOURS * 3600
while any(p.poll() is None for p in procs.values()) and time.time() < deadline:
    time.sleep(60)
for device, p in procs.items():
    if p.poll() is None:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        print(device, "stopped at deadline; will resume from checkpoint next version")
    else:
        print(device, "finished with code", p.returncode)

In [ ]:
!for f in /kaggle/working/log_*.txt; do echo == $f; tail -n 30 $f; done
!ls /kaggle/working/runs

In [ ]:
import glob, json
for path in sorted(glob.glob("/kaggle/working/runs/*/results.json")):
    r = json.load(open(path))
    print(f"{r['run_name']:45s} {r['final_val_loss']:.4f}")